# Etapa 2 · Qualidade de dados, continuação PLD e Open-Meteo

O que este notebook faz:

1. **Profiling** de todas as tabelas de origem e das tabelas externas (`campo_real_energia.*` e `externo.*`), gravado em `qualidade.perfil_colunas`.
2. **Regras de qualidade** escritas como funções reutilizáveis (uma por regra, registradas por dimensão), executadas **antes** da limpeza e gravadas em `qualidade.metricas_qualidade`.
3. **Decisão justificada** por problema (`corrigir`, `descartar`/`quarentena` ou `sinalizar`), em `qualidade.decisoes_qualidade`.
4. **Limpeza no caminho até o warehouse**: gera `qualidade.usina_tratada`, `parada_tratada` e `medicao_geracao_tratada`, mais `qualidade.quarentena`. As tabelas de origem **não são alteradas**.
5. As **mesmas regras** rodam de novo sobre as tabelas tratadas (**depois**), com as mesmas funções, só trocando os nomes das tabelas.
6. **Impacto nos indicadores de negócio** (geração, receita valorada ao PLD, preço capturado, fator de capacidade) antes × depois, em `qualidade.impacto_negocio`.
7. **Índice de qualidade** por dimensão e por tabela, para a pergunta 7 do painel.


In [1]:
%pip install -q psycopg2-binary pyyaml pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Configuração, conexão e tabelas de qualidade

In [2]:
import os, re, json, getpass
from pathlib import Path
from datetime import datetime, date, timezone

import yaml, numpy as np, pandas as pd, psycopg2
from psycopg2.extras import execute_values, Json

CONFIG_PATH = Path("config.yaml")

BASE_YAML = """
banco:
  variavel_ambiente: NEON_DATABASE_URL
  arquivo_env: .env
"""

QUALIDADE_YAML = """
qualidade:
  periodo_inicio: "2025-01-01 00:00:00"   # período que a medição de geração deve cobrir
  periodo_fim: "2025-12-31 23:00:00"
  atraso_max_horas: 24                    # recebimento depois disso conta como atrasado
  tolerancia_parada_envio_horas: 24       # usina sem dados há mais que isso "parou de enviar"
  sequencia_travada_min_horas: 6          # horas seguidas com o mesmo valor positivo
  janela_benchmark_dias: 7                # vizinhança usada para validar a correção de unidade
  tolerancia_correcao: 0.5                # valor corrigido deve ficar a +-50% da mediana da vizinhança
  limites_brasil: {lat_min: -34.0, lat_max: 5.5, lon_min: -74.0, lon_max: -34.0}
  pld_limites:                            # ANEEL: PLD minimo e maximo horario (R$/MWh)
    2025: {min: 58.60, max_horario: 1542.23}
    2026: {min: 57.31, max_horario: 1611.04}
"""

def mesclar(padrao, atual):
    for k, v in (atual or {}).items():
        if isinstance(v, dict) and isinstance(padrao.get(k), dict):
            mesclar(padrao[k], v)
        else:
            padrao[k] = v
    return padrao

cfg = yaml.safe_load(BASE_YAML + QUALIDADE_YAML)
if CONFIG_PATH.exists():
    atual = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    if "qualidade" not in atual:
        with open(CONFIG_PATH, "a", encoding="utf-8") as f:
            f.write("\n" + QUALIDADE_YAML.strip() + "\n")
        print("Seção 'qualidade' adicionada ao config.yaml existente.")
    cfg = mesclar(cfg, atual)
else:
    CONFIG_PATH.write_text((BASE_YAML + QUALIDADE_YAML).strip() + "\n", encoding="utf-8")

B, Q = cfg["banco"], cfg["qualidade"]
INI, FIM = Q["periodo_inicio"], Q["periodo_fim"]
LIM = Q["limites_brasil"]
print("período esperado da medição:", INI, "->", FIM)


COD_RE = "^CRE-[A-Z]{3}-[0-9]{3}$"          # padrão de código de usina (SQL e Python)

# Submercado -> UFs que pertencem a ele 
SUBMERCADO_UF = {
    "SE": ["SP", "RJ", "MG", "ES", "GO", "DF", "MT", "MS", "AC", "RO"],
    "S":  ["PR", "SC", "RS"],
    "NE": ["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI"],
    "N":  ["PA", "TO", "MA", "AM", "AP", "RR"],
}

# Correções que exigem conhecimento externo ao dado
CORRECOES_MANUAIS = {
    "CRE-BIO-005": {"uf": ("SP", "município Sertãozinho é paulista; coordenadas (-21,14; -47,99) estão em SP; "
                                 "submercado SE; PE pertence ao NE")},
}

Seção 'qualidade' adicionada ao config.yaml existente.
período esperado da medição: 2025-01-01 00:00:00 -> 2025-12-31 23:00:00


In [3]:
def ler_url_banco():
    var = B["variavel_ambiente"]
    if os.environ.get(var):
        return os.environ[var]
    env = Path(B["arquivo_env"])
    if env.exists():
        for linha in env.read_text(encoding="utf-8").splitlines():
            linha = linha.strip()
            if linha and not linha.startswith("#") and "=" in linha:
                k, v = linha.split("=", 1)
                if k.strip() == var:
                    return v.strip().strip('"').strip("'")
    return getpass.getpass(f"Cole a connection string do Neon ({var}): ")

conn = psycopg2.connect(ler_url_banco(), connect_timeout=30,
                        keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=5)

def consulta(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

def q(sql):
    """Executa uma consulta que devolve (avaliados, falhas)."""
    with conn.cursor() as cur:
        cur.execute(sql)
        a, f = cur.fetchone()
    return int(a or 0), int(f or 0)

def py(v):
    """Converte valores do pandas/numpy para tipos que o psycopg2 aceita."""
    if v is None:
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return None if np.isnan(v) else float(v)
    if isinstance(v, pd.Timestamp):
        return None if pd.isna(v) else v.to_pydatetime()
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v

with conn.cursor() as cur:
    cur.execute("SELECT to_regclass('externo.pld_horario'), to_regclass('externo.clima_horario'), to_regclass('campo_real_energia.medicao_geracao')")
    faltam = [n for n, ok in zip(["externo.pld_horario", "externo.clima_horario", "campo_real_energia.medicao_geracao"], cur.fetchone()) if ok is None]
if faltam:
    raise RuntimeError(f"Tabelas ausentes: {faltam}. Rode antes os notebooks do PLD e do Open-Meteo.")
print("Conectado.")

Conectado.


In [4]:
DDL = """
CREATE SCHEMA IF NOT EXISTS qualidade;

CREATE TABLE IF NOT EXISTS qualidade.metricas_qualidade (
  id_metrica          BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_execucao         INTEGER      NOT NULL,
  fase                VARCHAR(10)  NOT NULL,        -- ANTES | DEPOIS
  id_regra            VARCHAR(6)   NOT NULL,
  regra               VARCHAR(160) NOT NULL,
  dimensao            VARCHAR(30)  NOT NULL,
  tabela              VARCHAR(60)  NOT NULL,
  registros_avaliados BIGINT,
  registros_falha     BIGINT,
  percentual_falha    NUMERIC(9,4),
  erro                TEXT,
  executado_em        TIMESTAMPTZ  NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS qualidade.decisoes_qualidade (
  id_regra      VARCHAR(6) PRIMARY KEY,
  problema      TEXT        NOT NULL,
  decisao       VARCHAR(14) NOT NULL,             -- CORRIGIR | QUARENTENA | SINALIZAR | SEM_OCORRENCIA
  justificativa TEXT        NOT NULL,
  registrado_em TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS qualidade.perfil_colunas (
  fase           VARCHAR(10) NOT NULL,            -- ORIGEM | TRATADA
  tabela         VARCHAR(70) NOT NULL,
  coluna         VARCHAR(60) NOT NULL,
  tipo           VARCHAR(30),
  registros      BIGINT,
  nulos          BIGINT,
  pct_nulos      NUMERIC(9,3),
  distintos      BIGINT,
  minimo         TEXT,
  maximo         TEXT,
  media          DOUBLE PRECISION,
  mediana        DOUBLE PRECISION,
  p01            DOUBLE PRECISION,
  p99            DOUBLE PRECISION,
  mais_frequente TEXT,
  executado_em   TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS qualidade.quarentena (
  id_quarentena BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  tabela_origem VARCHAR(40) NOT NULL,
  id_registro   VARCHAR(40) NOT NULL,
  motivo        VARCHAR(60) NOT NULL,
  id_regra      VARCHAR(6),
  detalhe       JSONB,
  registrado_em TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS qualidade.impacto_negocio (
  nivel        VARCHAR(10) NOT NULL,               -- total | fonte | usina
  chave        VARCHAR(40) NOT NULL,
  indicador    VARCHAR(30) NOT NULL,
  antes        DOUBLE PRECISION,
  depois       DOUBLE PRECISION,
  delta        DOUBLE PRECISION,
  delta_pct    DOUBLE PRECISION,
  executado_em TIMESTAMPTZ NOT NULL DEFAULT now()
);
"""
with conn.cursor() as cur:
    cur.execute(DDL)
conn.commit()
print("Schema qualidade pronto.")

Schema qualidade pronto.


## 2. Profiling

Para cada coluna de cada tabela: tipo, nulos, distintos, mínimo, máximo, média, mediana, percentis 1% e 99% (distribuição) e o valor mais frequente. O resultado fica em `qualidade.perfil_colunas`. As tabelas `raw_*` (JSON) ficam de fora porque o conteúdo útil delas já está nas tabelas tratadas.

In [5]:
from decimal import Decimal

def _como_numero(s):
    if pd.api.types.is_bool_dtype(s):
        return None
    if pd.api.types.is_numeric_dtype(s):
        return s.astype("float64")
    nn = s.dropna().head(500)
    if len(nn) and all(isinstance(v, (Decimal, int, float)) and not isinstance(v, bool) for v in nn):
        return pd.to_numeric(s.map(lambda v: np.nan if pd.isna(v) else float(v)), errors="coerce")
    return None

def perfil_tabela(tabela, fase):
    df = consulta(f"SELECT * FROM {tabela}")
    linhas = []
    for c in df.columns:
        s = df[c]
        n = len(s)
        nulos = int(s.isna().sum())
        num = _como_numero(s)
        try:
            distintos = int(s.nunique(dropna=True))
        except TypeError:
            distintos = None
        try:
            mn, mx = s.min(), s.max()
        except TypeError:
            mn = mx = None
        moda = s.mode(dropna=True) if nulos < n else s.iloc[0:0]
        tem_num = num is not None and num.notna().any()
        linhas.append((fase, tabela, c, str(s.dtype), n, nulos, round(100 * nulos / n, 3) if n else None, distintos,
                       None if mn is None or pd.isna(mn) else str(mn), None if mx is None or pd.isna(mx) else str(mx),
                       float(num.mean()) if tem_num else None, float(num.median()) if tem_num else None,
                       float(num.quantile(0.01)) if tem_num else None, float(num.quantile(0.99)) if tem_num else None,
                       str(moda.iloc[0]) if len(moda) else None))
    return linhas

def gravar_perfil(tabelas, fase):
    todas = []
    for t in tabelas:
        todas += perfil_tabela(t, fase)
    with conn.cursor() as cur:
        cur.execute("DELETE FROM qualidade.perfil_colunas WHERE fase=%s", (fase,))
        execute_values(cur, """INSERT INTO qualidade.perfil_colunas
            (fase, tabela, coluna, tipo, registros, nulos, pct_nulos, distintos, minimo, maximo,
             media, mediana, p01, p99, mais_frequente) VALUES %s""", todas, page_size=500)
    conn.commit()

TABELAS_PERFIL = ["campo_real_energia.submercado", "campo_real_energia.fonte_energia", "campo_real_energia.usina",
                  "campo_real_energia.feriado_nacional", "campo_real_energia.parada_usina",
                  "campo_real_energia.medicao_geracao", "campo_real_energia.log_carga",
                  "externo.pld_horario", "externo.clima_horario", "externo.clima_ponto"]
gravar_perfil(TABELAS_PERFIL, "ORIGEM")

perfil = consulta("SELECT * FROM qualidade.perfil_colunas WHERE fase='ORIGEM' ORDER BY tabela, coluna")
print("Perfil gravado:", len(perfil), "colunas em", perfil["tabela"].nunique(), "tabelas")
display(perfil[perfil["tabela"].isin(["campo_real_energia.medicao_geracao", "campo_real_energia.usina"])]
        [["tabela", "coluna", "registros", "nulos", "pct_nulos", "distintos", "minimo", "maximo", "mediana", "p01", "p99", "mais_frequente"]])

Perfil gravado: 74 colunas em 10 tabelas


,tabela,coluna,registros,nulos,pct_nulos,distintos,minimo,maximo,mediana,p01,p99,mais_frequente
15,campo_real_energia.medicao_geracao,data_hora,78798,0,0.000,8761,2025-01-01 00:00:00,2052-05-14 10:00:00,NaN,NaN,NaN,2025-01-20 00:00:00
16,campo_real_energia.medicao_geracao,disponibilidade_mw,78798,1,0.001,10,0.00,180.00,90.000,0.00000,180.00000,25.00
17,campo_real_energia.medicao_geracao,energia_mwh,78798,6,0.008,44288,-0.850,33963.000,27.831,0.00000,135.17027,0.000
18,campo_real_energia.medicao_geracao,id_medicao,78798,0,0.000,78798,1,78894,39495.500,788.97000,78106.03000,1
19,campo_real_energia.medicao_geracao,id_usina,78798,1,0.001,10,1.0,99.0,5.000,1.00000,9.00000,2.0
20,campo_real_energia.medicao_geracao,origem,78798,0,0.000,2,MANUAL,SCADA,NaN,NaN,NaN,SCADA
21,campo_real_energia.medicao_geracao,recebido_em,78798,0,0.000,71144,2025-01-01 01:07:00,2026-02-10 09:23:00,NaN,NaN,NaN,2026-02-10 09:00:00
31,campo_real_energia.usina,cod_submercado,10,0,0.000,4,N,SE,NaN,NaN,NaN,NE
32,campo_real_energia.usina,cod_usina,10,0,0.000,10,CRE-BIO-005,CRE-UTE-009,NaN,NaN,NaN,CRE-BIO-005
33,campo_real_energia.usina,data_inicio_operacao,10,0,0.000,9,2006-05-01,2031-02-01,NaN,NaN,NaN,2015-09-01


In [6]:
print("Colunas com nulos (todas as tabelas):")
display(perfil[perfil["nulos"] > 0][["tabela", "coluna", "registros", "nulos", "pct_nulos"]])

print("\nMedição por origem do dado:")
display(consulta("SELECT origem, COUNT(*) AS linhas FROM campo_real_energia.medicao_geracao GROUP BY 1 ORDER BY 2 DESC"))

Colunas com nulos (todas as tabelas):


,tabela,coluna,registros,nulos,pct_nulos
16,campo_real_energia.medicao_geracao,disponibilidade_mw,78798,1,0.001
17,campo_real_energia.medicao_geracao,energia_mwh,78798,6,0.008
19,campo_real_energia.medicao_geracao,id_usina,78798,1,0.001
59,externo.clima_ponto,elevacao_m,10,1,10.000
60,externo.clima_ponto,lat_grade,10,1,10.000
62,externo.clima_ponto,latitude_usada,10,1,10.000
63,externo.clima_ponto,lon_grade,10,1,10.000
65,externo.clima_ponto,longitude_usada,10,1,10.000
67,externo.clima_ponto,observacao,10,8,80.000



Medição por origem do dado:


,origem,linhas
0,SCADA,78768
1,MANUAL,30


## 3. Regras de qualidade (funções reutilizáveis)

Cada regra é uma função `fn(q, T)` registrada com `@regra(id, dimensao, tabela, descricao)`. Ela recebe `T`, um dicionário com os **nomes das tabelas**, e devolve `(registros_avaliados, registros_com_falha)`. Por isso a mesma regra roda sobre a origem (**ANTES**) e sobre as tabelas tratadas (**DEPOIS**), sem copiar código.

| Dimensão | Regras |
|---|---|
| Completude | C01 a C06 |
| Unicidade | U01 a U03 |
| Validade | V01 a V08 |
| Consistência | K01 a K07 (inclui cruzamentos entre tabelas) |
| Acurácia | A01 a A03 |
| Integridade referencial | R01 a R03 |
| Atualidade | T01 a T03 |

Se alguma regra der erro de SQL, a execução não para: o erro fica na coluna `erro` da tabela de métricas.

In [7]:
REGRAS = []

def regra(id_, dimensao, tabela, descricao):
    def deco(fn):
        REGRAS.append({"id": id_, "dimensao": dimensao, "tabela": tabela, "descricao": descricao, "fn": fn})
        return fn
    return deco

ATRASO_H = Q["atraso_max_horas"]
TOL_ENVIO_H = Q["tolerancia_parada_envio_horas"]
RUN_MIN = Q["sequencia_travada_min_horas"]
FONTE = "campo_real_energia.fonte_energia"

TABELAS_ANTES = {"medicao": "campo_real_energia.medicao_geracao", "usina": "campo_real_energia.usina",
                 "parada": "campo_real_energia.parada_usina", "pld": "externo.pld_horario",
                 "raw_pld": "externo.raw_pld_horario", "clima": "externo.clima_horario"}
TABELAS_DEPOIS = {**TABELAS_ANTES, "medicao": "qualidade.medicao_geracao_tratada",
                  "usina": "qualidade.usina_tratada", "parada": "qualidade.parada_tratada"}

# ---------------------------------------------------------------- COMPLETUDE
@regra("C01", "Completude", "medicao_geracao", "Energia (MWh) nula")
def c01(q, T):
    return q(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE energia_mwh IS NULL) FROM {T['medicao']}")

@regra("C02", "Completude", "medicao_geracao", "Medição sem usina (id_usina nulo)")
def c02(q, T):
    return q(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE id_usina IS NULL) FROM {T['medicao']}")

@regra("C03", "Completude", "medicao_geracao", "Horas faltantes (usinas válidas x horas do período)")
def c03(q, T):
    return q(f"""
        WITH esperado AS (
          SELECT u.id_usina, g.h
          FROM {T['usina']} u
          CROSS JOIN generate_series(TIMESTAMP '{INI}', TIMESTAMP '{FIM}', INTERVAL '1 hour') AS g(h)
          WHERE u.cod_usina ~ '{COD_RE}'
        )
        SELECT COUNT(*), COUNT(*) FILTER (WHERE m.id_usina IS NULL)
        FROM esperado e
        LEFT JOIN (SELECT DISTINCT id_usina, data_hora FROM {T['medicao']}) m
               ON m.id_usina = e.id_usina AND m.data_hora = e.h""")

@regra("C04", "Completude", "usina", "Cadastro com nome, município ou UF vazio")
def c04(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE length(trim(COALESCE(nome, ''))) = 0
                                               OR length(trim(COALESCE(municipio, ''))) = 0
                                               OR length(trim(COALESCE(uf, ''))) = 0)
                 FROM {T['usina']}""")

@regra("C05", "Completude", "pld_horario", "Horas faltantes do PLD por submercado")
def c05(q, T):
    return q(f"""
        WITH r AS (SELECT cod_submercado, MIN(data_hora) AS mn, MAX(data_hora) AS mx, COUNT(*) AS n
                   FROM {T['pld']} GROUP BY 1)
        SELECT COALESCE(SUM((EXTRACT(EPOCH FROM (mx - mn)) / 3600)::bigint + 1), 0),
               COALESCE(SUM((EXTRACT(EPOCH FROM (mx - mn)) / 3600)::bigint + 1 - n), 0) FROM r""")

@regra("C06", "Completude", "clima_horario", "Horas faltantes do clima por usina")
def c06(q, T):
    return q(f"""
        WITH r AS (SELECT cod_usina, MIN(data_hora) AS mn, MAX(data_hora) AS mx, COUNT(*) AS n
                   FROM {T['clima']} GROUP BY 1)
        SELECT COALESCE(SUM((EXTRACT(EPOCH FROM (mx - mn)) / 3600)::bigint + 1), 0),
               COALESCE(SUM((EXTRACT(EPOCH FROM (mx - mn)) / 3600)::bigint + 1 - n), 0) FROM r""")

# ---------------------------------------------------------------- UNICIDADE
@regra("U01", "Unicidade", "medicao_geracao", "Chave (usina, hora) repetida")
def u01(q, T):
    return q(f"""
        SELECT (SELECT COUNT(*) FROM {T['medicao']} WHERE id_usina IS NOT NULL),
               (SELECT COALESCE(SUM(n - 1), 0) FROM (
                    SELECT COUNT(*) AS n FROM {T['medicao']} WHERE id_usina IS NOT NULL
                    GROUP BY id_usina, data_hora HAVING COUNT(*) > 1) x)""")

@regra("U02", "Unicidade", "usina", "Mesma usina cadastrada duas vezes (fonte, submercado, coordenadas, potência)")
def u02(q, T):
    return q(f"""
        SELECT (SELECT COUNT(*) FROM {T['usina']}),
               (SELECT COALESCE(SUM(n - 1), 0) FROM (
                    SELECT COUNT(*) AS n FROM {T['usina']}
                    GROUP BY id_fonte, cod_submercado, latitude, longitude, potencia_mw HAVING COUNT(*) > 1) x)""")

@regra("U03", "Unicidade", "raw_pld_horario", "PLD repetido (mês, dia, hora, submercado) na camada raw")
def u03(q, T):
    return q(f"""
        SELECT COUNT(*),
               COUNT(*) - COUNT(DISTINCT concat_ws('|', payload->>'MES_REFERENCIA', payload->>'DIA',
                                                         payload->>'HORA', payload->>'SUBMERCADO'))
        FROM {T['raw_pld']}""")

# ---------------------------------------------------------------- VALIDADE
@regra("V01", "Validade", "medicao_geracao", "Energia negativa")
def v01(q, T):
    return q(f"SELECT COUNT(energia_mwh), COUNT(*) FILTER (WHERE energia_mwh < 0) FROM {T['medicao']}")

@regra("V02", "Validade", "medicao_geracao", "Data/hora fora do período do cadastro ou fora da hora cheia")
def v02(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE data_hora < TIMESTAMP '{INI}' OR data_hora > TIMESTAMP '{FIM}'
                                               OR date_trunc('hour', data_hora) <> data_hora)
                 FROM {T['medicao']} WHERE data_hora IS NOT NULL""")

@regra("V03", "Validade", "medicao_geracao", "Disponibilidade fora de [0, potência instalada]")
def v03(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE m.disponibilidade_mw < 0 OR m.disponibilidade_mw > u.potencia_mw)
                 FROM {T['medicao']} m JOIN {T['usina']} u ON u.id_usina = m.id_usina
                 WHERE m.disponibilidade_mw IS NOT NULL""")

@regra("V04", "Validade", "usina", "Coordenadas fora do Brasil")
def v04(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE latitude NOT BETWEEN {LIM['lat_min']} AND {LIM['lat_max']}
                                               OR longitude NOT BETWEEN {LIM['lon_min']} AND {LIM['lon_max']})
                 FROM {T['usina']}""")

@regra("V05", "Validade", "usina", "Início de operação no futuro")
def v05(q, T):
    return q(f"SELECT COUNT(data_inicio_operacao), COUNT(*) FILTER (WHERE data_inicio_operacao > CURRENT_DATE) FROM {T['usina']}")

@regra("V06", "Validade", "parada_usina", "Parada com fim anterior ao início")
def v06(q, T):
    return q(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE fim < inicio) FROM {T['parada']}")

@regra("V07", "Validade", "pld_horario", "PLD fora dos limites regulatórios da ANEEL (mínimo e máximo horário do ano)")
def v07(q, T):
    vals = ", ".join(f"({ano}, {v['min']}, {v['max_horario']})" for ano, v in Q["pld_limites"].items())
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE p.pld_rs_mwh < l.pmin OR p.pld_rs_mwh > l.pmax)
                 FROM {T['pld']} p
                 JOIN (VALUES {vals}) AS l(ano, pmin, pmax) ON l.ano = EXTRACT(YEAR FROM p.data_hora)""")

@regra("V08", "Validade", "clima_horario", "Variável climática fora da faixa física")
def v08(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE temperatura_c NOT BETWEEN -20 AND 55
                    OR radiacao_w_m2 NOT BETWEEN 0 AND 1500 OR vento_10m_ms NOT BETWEEN 0 AND 60
                    OR vento_100m_ms NOT BETWEEN 0 AND 80 OR precipitacao_mm NOT BETWEEN 0 AND 300
                    OR cobertura_nuvens_pct NOT BETWEEN 0 AND 100 OR dir_vento_100m_graus NOT BETWEEN 0 AND 360)
                 FROM {T['clima']}""")

# ---------------------------------------------------------------- CONSISTÊNCIA
@regra("K01", "Consistência", "usina", "UF incompatível com o submercado da usina")
def k01(q, T):
    vals = ", ".join(f"('{s}', '{u}')" for s, ufs in SUBMERCADO_UF.items() for u in ufs)
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE NOT EXISTS (
                    SELECT 1 FROM (VALUES {vals}) AS m(sub, uf) WHERE m.sub = u.cod_submercado AND m.uf = u.uf))
                 FROM {T['usina']} u""")

@regra("K02", "Consistência", "medicao x parada", "Geração (>0) durante uma parada registrada")
def k02(q, T):
    return q(f"""SELECT (SELECT COUNT(*) FROM {T['medicao']} WHERE energia_mwh > 0),
                        (SELECT COUNT(DISTINCT m.id_medicao) FROM {T['medicao']} m
                         JOIN {T['parada']} p ON p.id_usina = m.id_usina AND m.data_hora BETWEEN p.inicio AND p.fim
                         WHERE m.energia_mwh > 0 AND p.fim >= p.inicio)""")

@regra("K03", "Consistência", "medicao x clima", "Usina solar com geração (>0) sem radiação solar")
def k03(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE c.radiacao_w_m2 <= 1)
                 FROM {T['medicao']} m
                 JOIN {T['usina']} u ON u.id_usina = m.id_usina
                 JOIN {FONTE} f ON f.id_fonte = u.id_fonte AND f.cod_fonte = 'UFV'
                 JOIN {T['clima']} c ON c.cod_usina = u.cod_usina AND c.data_hora = m.data_hora
                 WHERE m.energia_mwh > 0""")

@regra("K04", "Consistência", "medicao_geracao", "Recebido antes de acontecer (recebido_em < data_hora)")
def k04(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE recebido_em < data_hora)
                 FROM {T['medicao']} WHERE recebido_em IS NOT NULL AND data_hora IS NOT NULL""")

@regra("K05", "Consistência", "medicao x usina", "Medição anterior ao início de operação da usina")
def k05(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE m.data_hora::date < u.data_inicio_operacao)
                 FROM {T['medicao']} m JOIN {T['usina']} u ON u.id_usina = m.id_usina""")

@regra("K06", "Consistência", "raw_pld_horario", "PERIODO_COMERCIALIZACAO diferente de (DIA-1)*24 + HORA + 1")
def k06(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE ((payload->>'DIA')::int - 1) * 24 + (payload->>'HORA')::int + 1
                                                  <> (payload->>'PERIODO_COMERCIALIZACAO')::int)
                 FROM {T['raw_pld']}""")

@regra("K07", "Consistência", "clima_horario", "Radiação solar > 0 entre 0h e 3h")
def k07(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE radiacao_w_m2 > 0)
                 FROM {T['clima']} WHERE EXTRACT(HOUR FROM data_hora) BETWEEN 0 AND 3""")

# ---------------------------------------------------------------- ACURÁCIA
@regra("A01", "Acurácia", "medicao_geracao", "Energia na hora acima da potência instalada (vírgula ou unidade erradas)")
def a01(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE m.energia_mwh > u.potencia_mw + 0.000001)
                 FROM {T['medicao']} m JOIN {T['usina']} u ON u.id_usina = m.id_usina
                 WHERE m.energia_mwh IS NOT NULL""")

@regra("A02", "Acurácia", "medicao_geracao", f"Medidor travado: {RUN_MIN}+ horas seguidas com o mesmo valor positivo")
def a02(q, T):
    return q(f"""
        WITH x AS (
          SELECT id_usina, energia_mwh,
                 data_hora - ROW_NUMBER() OVER (PARTITION BY id_usina, energia_mwh ORDER BY data_hora) * INTERVAL '1 hour' AS grp
          FROM (SELECT DISTINCT id_usina, data_hora, energia_mwh FROM {T['medicao']}
                WHERE energia_mwh > 0 AND id_usina IS NOT NULL) d)
        SELECT (SELECT COUNT(*) FROM {T['medicao']} WHERE energia_mwh > 0),
               COALESCE(SUM(n), 0)
        FROM (SELECT COUNT(*) AS n FROM x GROUP BY id_usina, energia_mwh, grp HAVING COUNT(*) >= {RUN_MIN}) r""")

@regra("A03", "Acurácia", "pld_horario", "PLD outlier (fora de 3 x IQR por submercado)")
def a03(q, T):
    return q(f"""
        WITH s AS (SELECT cod_submercado,
                          percentile_cont(0.25) WITHIN GROUP (ORDER BY pld_rs_mwh) AS q1,
                          percentile_cont(0.75) WITHIN GROUP (ORDER BY pld_rs_mwh) AS q3
                   FROM {T['pld']} GROUP BY 1)
        SELECT COUNT(*), COUNT(*) FILTER (WHERE p.pld_rs_mwh > s.q3 + 3 * (s.q3 - s.q1)
                                             OR p.pld_rs_mwh < s.q1 - 3 * (s.q3 - s.q1))
        FROM {T['pld']} p JOIN s USING (cod_submercado)""")

# ---------------------------------------------------------------- INTEGRIDADE REFERENCIAL
@regra("R01", "Integridade referencial", "medicao_geracao", "Medição com id_usina que não existe no cadastro")
def r01(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM {T['usina']} u WHERE u.id_usina = m.id_usina))
                 FROM {T['medicao']} m WHERE m.id_usina IS NOT NULL""")

@regra("R02", "Integridade referencial", "parada_usina", "Parada com id_usina que não existe no cadastro")
def r02(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM {T['usina']} u WHERE u.id_usina = p.id_usina))
                 FROM {T['parada']} p""")

@regra("R03", "Integridade referencial", "clima_horario", "Clima de usina que não existe no cadastro")
def r03(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM {T['usina']} u WHERE u.cod_usina = c.cod_usina))
                 FROM {T['clima']} c""")

# ---------------------------------------------------------------- ATUALIDADE
@regra("T01", "Atualidade", "medicao_geracao", f"Dado recebido mais de {ATRASO_H} h depois da hora medida")
def t01(q, T):
    return q(f"""SELECT COUNT(*), COUNT(*) FILTER (WHERE recebido_em - data_hora > INTERVAL '{ATRASO_H} hours')
                 FROM {T['medicao']} WHERE recebido_em IS NOT NULL AND data_hora IS NOT NULL""")

@regra("T02", "Atualidade", "medicao_geracao", f"Usina que parou de enviar (última leitura {TOL_ENVIO_H}+ h antes das demais)")
def t02(q, T):
    return q(f"""
        WITH v AS (SELECT id_usina FROM {T['usina']} WHERE cod_usina ~ '{COD_RE}'),
        ult AS (SELECT v.id_usina, MAX(m.data_hora) AS mx
                FROM v LEFT JOIN {T['medicao']} m
                       ON m.id_usina = v.id_usina AND m.data_hora BETWEEN TIMESTAMP '{INI}' AND TIMESTAMP '{FIM}'
                GROUP BY v.id_usina),
        g AS (SELECT MAX(mx) AS gmx FROM ult)
        SELECT COUNT(*), COUNT(*) FILTER (WHERE ult.mx IS NULL OR g.gmx - ult.mx > INTERVAL '{TOL_ENVIO_H} hours')
        FROM ult CROSS JOIN g""")

@regra("T03", "Atualidade", "pld_horario", "PLD defasado: último dado com mais de 2 dias")
def t03(q, T):
    return q(f"""SELECT 1, CASE WHEN MAX(data_hora) < CURRENT_DATE - INTERVAL '2 days' THEN 1 ELSE 0 END FROM {T['pld']}""")

print(len(REGRAS), "regras registradas:",
      pd.Series([r["dimensao"] for r in REGRAS]).value_counts().to_dict())

33 regras registradas: {'Validade': 8, 'Consistência': 7, 'Completude': 6, 'Unicidade': 3, 'Acurácia': 3, 'Integridade referencial': 3, 'Atualidade': 3}


## 4. Execução ANTES (sobre as tabelas de origem)

In [8]:
def executar_regras(fase, T):
    with conn.cursor() as cur:
        cur.execute("SELECT COALESCE(MAX(id_execucao), 0) + 1 FROM qualidade.metricas_qualidade")
        id_exec = cur.fetchone()[0]
    linhas = []
    for r in REGRAS:
        try:
            a, f = r["fn"](q, T)
            linhas.append((id_exec, fase, r["id"], r["descricao"][:160], r["dimensao"], r["tabela"], a, f,
                           round(100.0 * f / a, 4) if a else None, None))
        except Exception as e:
            conn.rollback()
            linhas.append((id_exec, fase, r["id"], r["descricao"][:160], r["dimensao"], r["tabela"], None, None, None,
                           f"{type(e).__name__}: {str(e)[:300]}"))
    with conn.cursor() as cur:
        execute_values(cur, """INSERT INTO qualidade.metricas_qualidade
            (id_execucao, fase, id_regra, regra, dimensao, tabela, registros_avaliados, registros_falha, percentual_falha, erro)
            VALUES %s""", linhas)
    conn.commit()
    return id_exec

def resumo_execucao(id_exec, so_falhas=True):
    df = consulta("""SELECT id_regra, dimensao, tabela, regra, registros_avaliados AS avaliados,
                            registros_falha AS falhas, percentual_falha AS pct_falha, erro
                     FROM qualidade.metricas_qualidade WHERE id_execucao=%s ORDER BY dimensao, id_regra""", (id_exec,))
    erros = df[df["erro"].notna()]
    if len(erros):
        print("REGRAS COM ERRO DE EXECUÇÃO:")
        display(erros[["id_regra", "erro"]])
    print(f"{(df['falhas'] > 0).sum()} regras com falha, {(df['falhas'] == 0).sum()} sem ocorrência, {len(erros)} com erro")
    return df[df["falhas"] > 0] if so_falhas else df

id_antes = executar_regras("ANTES", TABELAS_ANTES)
display(resumo_execucao(id_antes))

22 regras com falha, 11 sem ocorrência, 0 com erro


,id_regra,dimensao,tabela,regra,avaliados,falhas,pct_falha,erro
0,A01,Acurácia,medicao_geracao,Energia na hora acima da potência instalada (v...,78767,26,0.0330,None
1,A02,Acurácia,medicao_geracao,Medidor travado: 6+ horas seguidas com o mesmo...,62195,72,0.1158,None
2,A03,Acurácia,pld_horario,PLD outlier (fora de 3 x IQR por submercado),64992,111,0.1708,None
3,T01,Atualidade,medicao_geracao,Dado recebido mais de 24 h depois da hora medida,78798,316,0.4010,None
4,T02,Atualidade,medicao_geracao,Usina que parou de enviar (última leitura 24+ ...,9,1,11.1111,None
6,C01,Completude,medicao_geracao,Energia (MWh) nula,78798,6,0.0076,None
7,C02,Completude,medicao_geracao,Medição sem usina (id_usina nulo),78798,1,0.0013,None
8,C03,Completude,medicao_geracao,Horas faltantes (usinas válidas x horas do per...,78840,96,0.1218,None
9,C04,Completude,usina,"Cadastro com nome, município ou UF vazio",10,1,10.0000,None
12,K01,Consistência,usina,UF incompatível com o submercado da usina,10,1,10.0000,None


## 5. Decisão justificada por problema

Cada regra que acusa falha tem uma decisão registrada em `qualidade.decisoes_qualidade`:

- **CORRIGIR**: só quando há evidência no próprio dado ou em fonte externa documentada, e o valor original é preservado (coluna `*_original` ou texto em `flags`).
- **QUARENTENA**: o registro sai da camada tratada e vai para `qualidade.quarentena`, com motivo e conteúdo original. Nada é apagado.
- **SINALIZAR**: o dado segue para o warehouse, marcado em `flags`; a ressalva é reportada.

 **Corrigir sem justificar é inventar dado**. 

In [9]:
DECISOES = {
 "C01": ("UHE Tocantins Real sem energia em 05/03 (10h a 15h): falha de comunicação do medidor.",
         "SINALIZAR", "Não há como saber a energia real; imputar seria inventar dado. A linha segue com energia nula e flag 'energia_ausente'. Nos indicadores o nulo não soma (equivale a zero), então a geração e a receita do período ficam subestimadas; o impacto é medido na seção 8."),
 "C02": ("Medição de 20/10 sem usina (id_usina nulo).",
         "QUARENTENA", "Não dá para atribuir a energia a nenhuma usina; entrar no fato distorceria totais de qualquer usina."),
 "C03": ("Horas ausentes: dia 10/11 da EOL Coxilha Real e fim de dezembro da UFV Campo Real Minas.",
         "SINALIZAR", "Horas não são recriadas. O fator de capacidade e a receita desses períodos refletem só o medido. Depois da limpeza as horas faltantes AUMENTAM porque linhas em quarentena deixam de existir; isso é esperado e não é imputado."),
 "C04": ("Cadastro com município vazio (duplicata da EOL Ventos do Potiguar).",
         "QUARENTENA", "Resolvido pela regra U02: o registro incompleto é justamente a duplicata."),
 "U01": ("Reprocessamento da integração duplicou 24 h da UFV Campo Real Minas; lançamento MANUAL (+10%) sobre leitura automática da PCH Rio das Araucárias.",
         "QUARENTENA", "Mantém-se uma linha por (usina, hora): a leitura SCADA tem prioridade sobre a MANUAL, e entre iguais fica a recebida primeiro. O lançamento manual chegou 2 dias depois, sem justificativa, e muda o valor em 10%."),
 "U02": ("Usina cadastrada duas vezes: CRE-EOL-07 (nome com espaço sobrando, município vazio) repete a CRE-EOL-007.",
         "QUARENTENA", "Mesmo ativo (fonte, submercado, coordenadas e potência). Fica o registro com código no padrão CRE-XXX-000; o duplicado vai para quarentena, sem medições associadas."),
 "V01": ("Energia negativa (-0,85 MWh) na EOL Ventos do Potiguar em 03/02, de 1h às 4h: consumo de serviços auxiliares lançado como geração.",
         "CORRIGIR", "Geração bruta não pode ser negativa; o valor negativo é consumo da usina parada, não geração. Convertido para 0 MWh, com o original guardado em energia_mwh_original e flag 'energia_negativa_zerada'."),
 "V02": ("Medição da UFV Sertão do Sol com data em 2052 (erro de digitação no ano).",
         "QUARENTENA", "Corrigir para 2025 criaria uma chave (usina, hora) que já existe com leitura automática. Sem como saber qual está certa, a linha vai para quarentena."),
 "V04": ("UFV Campo Real Minas com latitude +15,803 (cai no oceano Atlântico).",
         "CORRIGIR", "Só o sinal difere do ponto de Janaúba/MG; a inversão cai dentro do Brasil e a célula de clima devolvida (elevação 535 m) confirma terra firme. Original registrado em flags."),
 "V05": ("UFV Sertão do Sol com início de operação em 2031, mas com medições em 2025.",
         "CORRIGIR", "O valor correto é desconhecido. A data inválida é anulada (vira 'desconhecido' na dimensão) e o original fica em flags; as medições são mantidas."),
 "V06": ("Parada da UFV Campo Real Minas em 05/11 com fim (08h) antes do início (18h).",
         "QUARENTENA", "Não dá para saber qual dos dois horários está errado; inverter seria chutar. Fica fora das análises de perda por parada até alguém confirmar."),
 "K01": ("UTE Bagaço Campo Real (Sertãozinho) cadastrada com UF = PE, mas no submercado SE.",
         "CORRIGIR", "Correção manual documentada: Sertãozinho é paulista, as coordenadas estão em SP e PE pertence ao NE. UF ajustada para SP; original em flags."),
 "K02": ("UTE Gás Norte com 132 MWh de geração em 05/06 (18h a 21h), dentro da parada programada de 02 a 15/06.",
         "QUARENTENA", "A disponibilidade registrada nessas horas é 0 MW, o que corrobora a parada; a leitura de geração é incompatível e sai da camada tratada."),
 "K03": ("UFV Sertão do Sol com 15,4 MWh de geração entre 20h e 23h em 12/08 (sem radiação): relógio do medidor errado.",
         "QUARENTENA", "Fisicamente impossível. Não dá para deslocar as horas com segurança; a energia não é atribuída a nenhum horário."),
 "K04": ("PCH Rio das Araucárias com recebido_em 3 dias antes da hora medida em 05/08 (relógio do servidor).",
         "SINALIZAR", "O carimbo de recebimento está errado, mas a energia medida é plausível. A linha segue com flag 'recebido_antes_do_fato'."),
 "K05": ("Medições de 2025 da UFV Sertão do Sol anteriores à data de início cadastrada (2031).",
         "SINALIZAR", "O erro está no cadastro, não nas medições (a usina gerou o ano inteiro). Resolvido pela anulação da data (V05); as medições ficam."),
 "A01": ("Pico de UHE Campo Real Iguaçu (22/04 14h, x10), EOL Coxilha Real (499 MWh em 03/07 3h) e planilha em kWh da UTE Bagaço (10/05, x1000).",
         "CORRIGIR", "Se dividir por 10, 100 ou 1000 deixa o valor dentro da potência E a +-50% da mediana da mesma hora nos 7 dias vizinhos, corrige com flag; caso contrário (como os 499 MWh da eólica) vai para quarentena."),
 "A02": ("Medidor da UTE Bagaço travado em 31,25 MWh por 72 h (01 a 03/09).",
         "QUARENTENA", "Valor idêntico por 72 horas não ocorre em geração real. Fica fora; a perda de ~2.250 MWh de energia é quantificada no impacto, não imputada."),
 "A03": ("Valores de PLD acima de 3 x IQR (picos de preço).",
         "SINALIZAR", "Picos de PLD são sinal de mercado e estão dentro dos limites regulatórios (V07). Remover distorceria o preço capturado; ficam sinalizados."),
 "R01": ("Medições de um equipamento fora do cadastro (id_usina = 99).",
         "QUARENTENA", "Sem usina cadastrada, não há dimensão para ligar o fato."),
 "T01": ("Dados atrasados: UHE Tocantins Real a partir de 20/12 chegou só em fevereiro/2026; reprocessamentos e lançamentos manuais com 1 a 2 dias de atraso.",
         "SINALIZAR", "O dado é válido, só chegou tarde. Vale para o indicador de atualidade (SLA de integração), não para o valor da energia."),
 "T02": ("UFV Campo Real Minas parou de enviar dados a partir de 29/12.",
         "SINALIZAR", "Não se projeta geração. As 72 horas ausentes entram nas horas faltantes (C03) e na análise de disponibilidade."),
}

with conn.cursor() as cur:
    cur.execute("DELETE FROM qualidade.decisoes_qualidade")
    execute_values(cur, "INSERT INTO qualidade.decisoes_qualidade (id_regra, problema, decisao, justificativa) VALUES %s",
                   [(k, p, d, j) for k, (p, d, j) in DECISOES.items()])
conn.commit()

falhas_antes = consulta("""SELECT id_regra, dimensao, registros_falha FROM qualidade.metricas_qualidade
                           WHERE id_execucao=%s AND registros_falha > 0""", (id_antes,))
sem = falhas_antes[~falhas_antes["id_regra"].isin(DECISOES)]
print("Regras com falha SEM decisão registrada:", sem["id_regra"].tolist() or "nenhuma")
display(consulta("""SELECT d.id_regra, m.dimensao, m.registros_falha AS falhas, d.decisao, d.justificativa
                    FROM qualidade.decisoes_qualidade d
                    JOIN qualidade.metricas_qualidade m ON m.id_regra = d.id_regra AND m.id_execucao = %s
                    ORDER BY m.dimensao, d.id_regra""", (id_antes,)))

Regras com falha SEM decisão registrada: nenhuma


,id_regra,dimensao,falhas,decisao,justificativa
0,A01,Acurácia,26,CORRIGIR,"Se dividir por 10, 100 ou 1000 deixa o valor d..."
1,A02,Acurácia,72,QUARENTENA,Valor idêntico por 72 horas não ocorre em gera...
2,A03,Acurácia,111,SINALIZAR,Picos de PLD são sinal de mercado e estão dent...
3,T01,Atualidade,316,SINALIZAR,"O dado é válido, só chegou tarde. Vale para o ..."
4,T02,Atualidade,1,SINALIZAR,Não se projeta geração. As 72 horas ausentes e...
5,C01,Completude,6,SINALIZAR,Não há como saber a energia real; imputar seri...
6,C02,Completude,1,QUARENTENA,Não dá para atribuir a energia a nenhuma usina...
7,C03,Completude,96,SINALIZAR,Horas não são recriadas. O fator de capacidade...
8,C04,Completude,1,QUARENTENA,Resolvido pela regra U02: o registro incomplet...
9,K01,Consistência,1,CORRIGIR,Correção manual documentada: Sertãozinho é pau...


Confira a tabela acima: cada problema **que o script de origem plantou** deveria aparecer com falha. Se uma regra com decisão aparece com 0 falhas (ou o contrário), o resultado é um achado em si; me mande a tabela para eu ajustar.

## 6. Limpeza: do dado bruto para a camada tratada

As funções de tratamento são **puras** (recebem DataFrames e devolvem DataFrames), o que permite reaproveitá-las e testá-las. A ordem importa: cada linha da medição sai uma única vez, pelo **primeiro** motivo que a atinge.

1. sem usina → usina fora do cadastro → data fora do período
2. chave repetida (fica SCADA, depois a recebida primeiro)
3. energia negativa → zero (original preservado)
4. energia acima da potência → tenta ÷10/÷100/÷1000 validado pela vizinhança; senão quarentena
5. medidor travado → solar sem radiação → geração durante parada
6. só sinalização: energia ausente, recebimento atrasado, recebido antes do fato

In [10]:
import re
import numpy as np
import pandas as pd

def juntar(flags, novo):
    return f"{flags};{novo}" if flags else novo

def tratar_usinas(usi, hoje, lim, sub_uf, correcoes):
    """Cadastro de usinas -> (tratada, quarentena). Não altera o DataFrame de entrada."""
    u = usi.copy()
    for c in ("latitude", "longitude", "potencia_mw"):
        u[c] = u[c].astype("float64")
    u["flags"] = ""
    quarentena = []

    # 1) duplicidade de cadastro: mesmo ativo (fonte, submercado, coordenadas, potência) com dois códigos
    chave = ["id_fonte", "cod_submercado", "latitude", "longitude", "potencia_mw"]
    u["_padrao"] = u["cod_usina"].map(lambda c: bool(re.fullmatch(COD_RE, str(c))))
    ordem = u.sort_values(["_padrao", "id_usina"], ascending=[False, True])
    dup_idx = ordem.index[ordem.duplicated(chave, keep="first")]
    for i in dup_idx:
        det = {k: (None if pd.isna(v) else v) for k, v in u.loc[i].drop(["_padrao", "flags"]).to_dict().items()}
        quarentena.append(("usina", str(u.at[i, "id_usina"]), "cadastro_duplicado", "U02", det))
    u = u.drop(index=dup_idx)

    no_br = lambda la, lo: lim["lat_min"] <= la <= lim["lat_max"] and lim["lon_min"] <= lo <= lim["lon_max"]
    for i, r in u.iterrows():
        # 2) nome com espaços sobrando
        if isinstance(r["nome"], str) and r["nome"] != r["nome"].strip():
            u.at[i, "nome"] = r["nome"].strip()
            u.at[i, "flags"] = juntar(u.at[i, "flags"], "nome_aparado")
        # 3) latitude com sinal trocado (só corrige se a inversão cai dentro do Brasil)
        if not no_br(r["latitude"], r["longitude"]) and no_br(-r["latitude"], r["longitude"]):
            u.at[i, "flags"] = juntar(u.at[i, "flags"], f"latitude_sinal_invertido(original={r['latitude']})")
            u.at[i, "latitude"] = -r["latitude"]
        # 4) UF incompatível com o submercado
        if r["uf"] not in sub_uf.get(r["cod_submercado"], []):
            corr = correcoes.get(r["cod_usina"], {}).get("uf")
            if corr and corr[0] in sub_uf.get(r["cod_submercado"], []):
                u.at[i, "flags"] = juntar(u.at[i, "flags"], f"uf_corrigida(original={r['uf']})")
                u.at[i, "uf"] = corr[0]
            else:
                u.at[i, "flags"] = juntar(u.at[i, "flags"], "uf_incompativel_submercado")
        # 5) data de início de operação no futuro: não dá para saber a data certa, então fica nula
        d = r["data_inicio_operacao"]
        if pd.notna(d) and d > hoje:
            u.at[i, "flags"] = juntar(u.at[i, "flags"], f"data_inicio_futura_anulada(original={d})")
            u.at[i, "data_inicio_operacao"] = None
    return u.drop(columns=["_padrao"]), quarentena


def tratar_paradas(par, ids_validos):
    p = par.copy()
    p["inicio"] = pd.to_datetime(p["inicio"])
    p["fim"] = pd.to_datetime(p["fim"])
    p["flags"] = ""
    quarentena = []
    m1 = p["fim"] < p["inicio"]
    m2 = ~m1 & ~p["id_usina"].isin(ids_validos)
    for mask, motivo, regra in ((m1, "fim_antes_do_inicio", "V06"), (m2, "usina_inexistente", "R02")):
        for _, r in p[mask].iterrows():
            det = {k: (None if pd.isna(v) else str(v)) for k, v in r.drop("flags").to_dict().items()}
            quarentena.append(("parada_usina", str(r["id_parada"]), motivo, regra, det))
    return p[~(m1 | m2)].copy(), quarentena


def tratar_medicao(med, usi_t, par_t, clima, fonte_por_id, cod_por_id, P):
    """
    Medição bruta -> (tratada, quarentena[(id_medicao, motivo, id_regra)]).
    Ordem das regras: cada linha só é descartada uma vez, pelo primeiro motivo que a atinge.
    """
    m = med.copy()
    m["id_usina"] = pd.to_numeric(m["id_usina"], errors="coerce")
    for c in ("energia_mwh", "disponibilidade_mw"):
        m[c] = pd.to_numeric(m[c].astype("float64"), errors="coerce")
    m["data_hora"] = pd.to_datetime(m["data_hora"], errors="coerce")
    m["recebido_em"] = pd.to_datetime(m["recebido_em"], errors="coerce")
    m["energia_mwh_original"] = m["energia_mwh"]
    m["flags"] = ""
    q = []

    def tirar(df, mask, motivo, regra):
        q.extend((int(i), motivo, regra) for i in df.loc[mask, "id_medicao"])
        return df.loc[~mask].copy()

    def marcar(df, mask, texto):
        df.loc[mask, "flags"] = df.loc[mask, "flags"].map(lambda f: juntar(f, texto))
        return df

    ini, fim = pd.Timestamp(P["ini"]), pd.Timestamp(P["fim"])

    # C02: sem usina
    m = tirar(m, m["id_usina"].isna(), "usina_nula", "C02")
    m["id_usina"] = m["id_usina"].astype("int64")
    # R01: usina fora do cadastro (tratado)
    validas = {int(x) for x in usi_t["id_usina"]}
    m = tirar(m, ~m["id_usina"].isin(validas), "usina_inexistente", "R01")
    # V02: data fora do período ou fora da hora cheia
    fora = m["data_hora"].isna() | (m["data_hora"] < ini) | (m["data_hora"] > fim) | (m["data_hora"] != m["data_hora"].dt.floor("h"))
    m = tirar(m, fora, "data_fora_do_periodo", "V02")

    # U01: chave (usina, hora) repetida -> fica SCADA, depois o recebido primeiro
    prio = m["origem"].fillna("").map(lambda o: 0 if o == "SCADA" else 1)
    ordem = m.assign(_p=prio).sort_values(["id_usina", "data_hora", "_p", "recebido_em", "id_medicao"])
    dup_idx = ordem.index[ordem.duplicated(["id_usina", "data_hora"], keep="first")]
    eh_dup = m.index.isin(dup_idx)
    manual = m["origem"].fillna("").eq("MANUAL").to_numpy()
    m = tirar(m, pd.Series(eh_dup & manual, index=m.index), "lancamento_manual_sobre_leitura_automatica", "U01")
    m = tirar(m, pd.Series(m.index.isin(dup_idx), index=m.index), "duplicado_reprocessamento", "U01")

    # V01: energia negativa (consumo de serviços auxiliares) -> geração bruta zero, original preservado
    neg = m["energia_mwh"] < 0
    m = marcar(m, neg, "energia_negativa_zerada")
    m.loc[neg, "energia_mwh"] = 0.0

    # A01: energia acima da potência instalada (vírgula deslocada / unidade errada)
    pot = usi_t.set_index("id_usina")["potencia_mw"].astype("float64")
    m["_pot"] = m["id_usina"].map(pot)
    exc = m["energia_mwh"] > m["_pot"] + 1e-9
    base = m.loc[~exc & m["energia_mwh"].notna(), ["id_usina", "data_hora", "energia_mwh"]]
    base_por_usina = {k: g for k, g in base.groupby("id_usina")}
    rejeitar = []
    for idx in m.index[exc]:
        r = m.loc[idx]
        g = base_por_usina.get(r["id_usina"])
        novo = None
        if g is not None:
            viz = g[(g["data_hora"].dt.hour == r["data_hora"].hour)
                    & ((g["data_hora"] - r["data_hora"]).abs() <= pd.Timedelta(days=P["janela_dias"]))]
            mediana = viz["energia_mwh"].median()
            if pd.notna(mediana) and mediana > 0:
                for d in (10, 100, 1000):
                    cand = r["energia_mwh"] / d
                    if cand <= r["_pot"] and (1 - P["tol"]) * mediana <= cand <= (1 + P["tol"]) * mediana:
                        novo = (round(float(cand), 3), d)
                        break
        if novo:
            m.at[idx, "energia_mwh"] = novo[0]
            m.at[idx, "flags"] = juntar(m.at[idx, "flags"], f"energia_corrigida_div{novo[1]}")
        else:
            rejeitar.append(idx)
    m = tirar(m, pd.Series(m.index.isin(rejeitar), index=m.index), "energia_acima_potencia", "A01")

    # A02: medidor travado (>= N horas seguidas com o mesmo valor positivo)
    x = m[m["energia_mwh"] > 0].sort_values(["id_usina", "data_hora"])
    novo_grupo = ((x["id_usina"] != x["id_usina"].shift())
                  | (x["energia_mwh"] != x["energia_mwh"].shift())
                  | ((x["data_hora"] - x["data_hora"].shift()) != pd.Timedelta(hours=1)))
    grp = novo_grupo.cumsum()
    tam = x.groupby(grp)["energia_mwh"].transform("size")
    travados = x.index[tam >= P["run_min"]]
    m = tirar(m, pd.Series(m.index.isin(travados), index=m.index), "medidor_travado", "A02")

    # K03: usina solar gerando sem radiação (relógio do medidor errado)
    cod = m["id_usina"].map(cod_por_id)
    fonte = m["id_usina"].map(fonte_por_id)
    cl = clima.copy()
    cl["data_hora"] = pd.to_datetime(cl["data_hora"])
    rad = cl.drop_duplicates(["cod_usina", "data_hora"]).set_index(["cod_usina", "data_hora"])["radiacao_w_m2"].astype("float64")
    chave = pd.MultiIndex.from_arrays([cod.to_numpy(), m["data_hora"].to_numpy()])
    m["_rad"] = rad.reindex(chave).to_numpy()
    noturno = (fonte == "UFV") & (m["energia_mwh"] > 0) & (m["_rad"] <= 1)
    m = tirar(m, noturno.fillna(False), "solar_sem_radiacao", "K03")

    # K02: geração durante parada registrada
    em_parada = pd.Series(False, index=m.index)
    pt = par_t.copy()
    pt["inicio"] = pd.to_datetime(pt["inicio"])
    pt["fim"] = pd.to_datetime(pt["fim"])
    for _, p in pt.iterrows():
        em_parada |= ((m["id_usina"] == p["id_usina"]) & m["data_hora"].between(p["inicio"], p["fim"])
                      & (m["energia_mwh"] > 0))
    m = tirar(m, em_parada, "geracao_durante_parada", "K02")

    # Só sinalizar (dado válido, mas com ressalva)
    m = marcar(m, m["energia_mwh"].isna(), "energia_ausente")
    atrasado = (m["recebido_em"] - m["data_hora"]) > pd.Timedelta(hours=P["atraso_h"])
    m = marcar(m, atrasado.fillna(False), "recebimento_atrasado")
    m = marcar(m, (m["recebido_em"] < m["data_hora"]).fillna(False), "recebido_antes_do_fato")

    cols = ["id_medicao", "id_usina", "data_hora", "energia_mwh", "energia_mwh_original",
            "disponibilidade_mw", "origem", "recebido_em", "flags"]
    return m[cols].copy(), q

In [11]:
# --- carrega a origem e aplica o tratamento (nada é gravado na origem) ---
usi = consulta("SELECT * FROM campo_real_energia.usina ORDER BY id_usina")
par = consulta("SELECT * FROM campo_real_energia.parada_usina ORDER BY id_parada")
med = consulta("SELECT * FROM campo_real_energia.medicao_geracao")
clima = consulta("SELECT cod_usina, data_hora, radiacao_w_m2 FROM externo.clima_horario")
fontes = consulta("""SELECT u.id_usina, u.cod_usina, f.cod_fonte
                     FROM campo_real_energia.usina u JOIN campo_real_energia.fonte_energia f ON f.id_fonte = u.id_fonte""")
fonte_por_id = dict(zip(fontes["id_usina"], fontes["cod_fonte"]))
cod_por_id = dict(zip(fontes["id_usina"], fontes["cod_usina"]))

hoje = datetime.now(timezone.utc).date()
usi_t, q_usina = tratar_usinas(usi, hoje, LIM, SUBMERCADO_UF, CORRECOES_MANUAIS)
par_t, q_parada = tratar_paradas(par, set(int(x) for x in usi_t["id_usina"]))
P = {"ini": INI, "fim": FIM, "atraso_h": ATRASO_H, "run_min": RUN_MIN,
     "janela_dias": Q["janela_benchmark_dias"], "tol": Q["tolerancia_correcao"]}
med_t, q_med = tratar_medicao(med, usi_t, par_t, clima, fonte_por_id, cod_por_id, P)

print(f"usina:    {len(usi)} -> {len(usi_t)}  (quarentena {len(q_usina)})")
print(f"parada:   {len(par)} -> {len(par_t)}  (quarentena {len(q_parada)})")
print(f"medição:  {len(med)} -> {len(med_t)}  (quarentena {len(q_med)})")
display(pd.DataFrame(q_med, columns=["id_medicao", "motivo", "regra"]).groupby(["regra", "motivo"]).size().rename("linhas").reset_index())
print("\nLinhas tratadas com flag:")
display(med_t[med_t["flags"] != ""].assign(flag=lambda d: d["flags"].str.split(";")).explode("flag")
        .groupby("flag").size().rename("linhas").reset_index())
display(usi_t[usi_t["flags"] != ""][["cod_usina", "nome", "uf", "latitude", "data_inicio_operacao", "flags"]])

usina:    10 -> 9  (quarentena 1)
parada:   11 -> 10  (quarentena 1)
medição:  78798 -> 78663  (quarentena 135)


,regra,motivo,linhas
0,A01,energia_acima_potencia,1
1,A02,medidor_travado,72
2,C02,usina_nula,1
3,K02,geracao_durante_parada,4
4,K03,solar_sem_radiacao,4
5,R01,usina_inexistente,24
6,U01,duplicado_reprocessamento,24
7,U01,lancamento_manual_sobre_leitura_automatica,4
8,V02,data_fora_do_periodo,1



Linhas tratadas com flag:


,flag,linhas
0,energia_ausente,6
1,energia_corrigida_div10,1
2,energia_corrigida_div1000,24
3,energia_negativa_zerada,4
4,recebido_antes_do_fato,24
5,recebimento_atrasado,288


,cod_usina,nome,uf,latitude,data_inicio_operacao,flags
0,CRE-UHE-001,UHE Campo Real Iguaçu,PR,-25.480,2009-03-01,uf_incompativel_submercado
1,CRE-PCH-002,PCH Rio das Araucárias,SC,-26.775,2014-08-15,uf_incompativel_submercado
2,CRE-EOL-003,EOL Coxilha Real,RS,-30.890,2016-11-01,uf_incompativel_submercado
3,CRE-UFV-004,UFV Campo Real Minas,MG,-15.803,2020-06-01,latitude_sinal_invertido(original=15.803)
4,CRE-BIO-005,UTE Bagaço Campo Real,SP,-21.138,2012-04-01,uf_corrigida(original=PE)
5,CRE-UFV-006,UFV Sertão do Sol,BA,-13.255,None,data_inicio_futura_anulada(original=2031-02-01)
7,CRE-UHE-008,UHE Tocantins Real,TO,-12.025,2006-05-01,uf_incompativel_submercado
8,CRE-UTE-009,UTE Gás Norte,MA,-4.867,2013-01-01,uf_incompativel_submercado


In [12]:
# --- grava a camada tratada e a quarentena ---
DDL_TRATADA = """
DROP TABLE IF EXISTS qualidade.medicao_geracao_tratada, qualidade.parada_tratada, qualidade.usina_tratada;

CREATE TABLE qualidade.usina_tratada (
  id_usina INTEGER PRIMARY KEY, cod_usina VARCHAR(12) NOT NULL UNIQUE, nome VARCHAR(80) NOT NULL,
  id_fonte SMALLINT NOT NULL, cod_submercado CHAR(2) NOT NULL, uf CHAR(2) NOT NULL, municipio VARCHAR(60) NOT NULL,
  latitude NUMERIC(9,6) NOT NULL, longitude NUMERIC(9,6) NOT NULL, potencia_mw NUMERIC(8,2) NOT NULL,
  data_inicio_operacao DATE, situacao VARCHAR(20) NOT NULL, flags TEXT
);
CREATE TABLE qualidade.parada_tratada (
  id_parada INTEGER PRIMARY KEY, id_usina INTEGER NOT NULL, inicio TIMESTAMP NOT NULL, fim TIMESTAMP NOT NULL,
  tipo VARCHAR(20) NOT NULL, motivo VARCHAR(120) NOT NULL, flags TEXT
);
CREATE TABLE qualidade.medicao_geracao_tratada (
  id_medicao BIGINT PRIMARY KEY, id_usina INTEGER NOT NULL, data_hora TIMESTAMP NOT NULL,
  energia_mwh NUMERIC(12,3), energia_mwh_original NUMERIC(12,3), disponibilidade_mw NUMERIC(8,2),
  origem VARCHAR(20), recebido_em TIMESTAMP, flags TEXT,
  UNIQUE (id_usina, data_hora)
);
"""

def detalhe(d):
    return Json({k: (None if (not isinstance(v, (list, dict)) and pd.isna(v)) else v) for k, v in d.items()},
                dumps=lambda o: json.dumps(o, default=str))

med_idx = med.set_index("id_medicao")
quarentena_rows = [(t, i, m, r, detalhe(d)) for t, i, m, r, d in q_usina + q_parada]
quarentena_rows += [("medicao_geracao", str(i), m, r, detalhe(med_idx.loc[i].to_dict())) for i, m, r in q_med]

try:
    with conn.cursor() as cur:
        cur.execute(DDL_TRATADA)
        cur.execute("TRUNCATE qualidade.quarentena")
        cols_u = ["id_usina", "cod_usina", "nome", "id_fonte", "cod_submercado", "uf", "municipio", "latitude",
                  "longitude", "potencia_mw", "data_inicio_operacao", "situacao", "flags"]
        execute_values(cur, f"INSERT INTO qualidade.usina_tratada ({', '.join(cols_u)}) VALUES %s",
                       [tuple(py(v) for v in r) for r in usi_t[cols_u].itertuples(index=False, name=None)])
        cols_p = ["id_parada", "id_usina", "inicio", "fim", "tipo", "motivo", "flags"]
        execute_values(cur, f"INSERT INTO qualidade.parada_tratada ({', '.join(cols_p)}) VALUES %s",
                       [tuple(py(v) for v in r) for r in par_t[cols_p].itertuples(index=False, name=None)])
        cols_m = list(med_t.columns)
        execute_values(cur, f"INSERT INTO qualidade.medicao_geracao_tratada ({', '.join(cols_m)}) VALUES %s",
                       [tuple(py(v) for v in r) for r in med_t.itertuples(index=False, name=None)], page_size=2000)
        execute_values(cur, """INSERT INTO qualidade.quarentena (tabela_origem, id_registro, motivo, id_regra, detalhe)
                               VALUES %s""", quarentena_rows, page_size=500)
    conn.commit()
    print("Camada tratada gravada.")
except Exception:
    conn.rollback()
    raise

display(consulta("""SELECT tabela_origem, id_regra, motivo, COUNT(*) AS registros
                    FROM qualidade.quarentena GROUP BY 1,2,3 ORDER BY 1,2,3"""))

Camada tratada gravada.


,tabela_origem,id_regra,motivo,registros
0,medicao_geracao,A01,energia_acima_potencia,1
1,medicao_geracao,A02,medidor_travado,72
2,medicao_geracao,C02,usina_nula,1
3,medicao_geracao,K02,geracao_durante_parada,4
4,medicao_geracao,K03,solar_sem_radiacao,4
5,medicao_geracao,R01,usina_inexistente,24
6,medicao_geracao,U01,duplicado_reprocessamento,24
7,medicao_geracao,U01,lancamento_manual_sobre_leitura_automatica,4
8,medicao_geracao,V02,data_fora_do_periodo,1
9,parada_usina,V06,fim_antes_do_inicio,1


## 7. Execução DEPOIS (tabelas tratadas) e comparação

In [13]:
id_depois = executar_regras("DEPOIS", TABELAS_DEPOIS)
gravar_perfil(["qualidade.usina_tratada", "qualidade.parada_tratada", "qualidade.medicao_geracao_tratada"], "TRATADA")

comparacao = consulta("""
    WITH ult AS (SELECT fase, MAX(id_execucao) AS id_execucao FROM qualidade.metricas_qualidade GROUP BY fase)
    SELECT m.id_regra, m.dimensao, m.tabela, m.regra,
           MAX(CASE WHEN m.fase='ANTES'  THEN m.registros_falha END)   AS falhas_antes,
           MAX(CASE WHEN m.fase='DEPOIS' THEN m.registros_falha END)   AS falhas_depois,
           MAX(CASE WHEN m.fase='ANTES'  THEN m.percentual_falha END)  AS pct_antes,
           MAX(CASE WHEN m.fase='DEPOIS' THEN m.percentual_falha END)  AS pct_depois,
           d.decisao
    FROM qualidade.metricas_qualidade m
    JOIN ult u ON u.fase = m.fase AND u.id_execucao = m.id_execucao
    LEFT JOIN qualidade.decisoes_qualidade d ON d.id_regra = m.id_regra
    GROUP BY m.id_regra, m.dimensao, m.tabela, m.regra, d.decisao
    ORDER BY m.dimensao, m.id_regra""")
print("Regras com falha antes ou depois:")
display(comparacao[(comparacao["falhas_antes"] > 0) | (comparacao["falhas_depois"] > 0)])
print("\nRegras com erro de execução:", comparacao[comparacao["falhas_antes"].isna() | comparacao["falhas_depois"].isna()]["id_regra"].tolist() or "nenhuma")

Regras com falha antes ou depois:


,id_regra,dimensao,tabela,regra,falhas_antes,falhas_depois,pct_antes,pct_depois,decisao
0,A01,Acurácia,medicao_geracao,Energia na hora acima da potência instalada (v...,26,0,0.0330,0.0000,CORRIGIR
1,A02,Acurácia,medicao_geracao,Medidor travado: 6+ horas seguidas com o mesmo...,72,0,0.1158,0.0000,QUARENTENA
2,A03,Acurácia,pld_horario,PLD outlier (fora de 3 x IQR por submercado),111,111,0.1708,0.1708,SINALIZAR
3,T01,Atualidade,medicao_geracao,Dado recebido mais de 24 h depois da hora medida,316,288,0.4010,0.3661,SINALIZAR
4,T02,Atualidade,medicao_geracao,Usina que parou de enviar (última leitura 24+ ...,1,1,11.1111,11.1111,SINALIZAR
6,C01,Completude,medicao_geracao,Energia (MWh) nula,6,6,0.0076,0.0076,SINALIZAR
7,C02,Completude,medicao_geracao,Medição sem usina (id_usina nulo),1,0,0.0013,0.0000,QUARENTENA
8,C03,Completude,medicao_geracao,Horas faltantes (usinas válidas x horas do per...,96,177,0.1218,0.2245,SINALIZAR
9,C04,Completude,usina,"Cadastro com nome, município ou UF vazio",1,0,10.0000,0.0000,QUARENTENA
12,K01,Consistência,usina,UF incompatível com o submercado da usina,1,0,10.0000,0.0000,CORRIGIR



Regras com erro de execução: nenhuma


**Como ler:** o que foi corrigido ou colocado em quarentena deve cair para 0 depois. O que foi só sinalizado continua aparecendo (atualidade, recebimento fora de ordem, nulos, picos de PLD): é esperado, porque o dado segue para o warehouse com a ressalva. As horas faltantes (C03) sobem depois da limpeza: as linhas que foram para a quarentena viram buracos, e o notebook não os preenche.

## 8. Impacto nos indicadores de negócio (antes × depois)

- **Antes:** o que um analista calcularia ligando a medição bruta ao cadastro original e ao PLD (as linhas sem usina cadastrada ficam de fora pela própria junção).
- **Depois:** a mesma conta sobre as tabelas tratadas.

Indicadores: geração (MWh), receita valorada ao PLD, preço capturado (R$/MWh) e fator de capacidade (sobre 8.760 h). Resultado gravado em `qualidade.impacto_negocio`.

In [14]:
def indicadores(T):
    df = consulta(f"""
        SELECT u.cod_usina, f.cod_fonte, u.cod_submercado, u.potencia_mw::float AS potencia_mw,
               SUM(m.energia_mwh)::float                       AS geracao_mwh,
               SUM(m.energia_mwh * p.pld_rs_mwh)::float        AS receita_rs
        FROM {T['medicao']} m
        JOIN {T['usina']} u ON u.id_usina = m.id_usina
        JOIN {FONTE} f ON f.id_fonte = u.id_fonte
        LEFT JOIN externo.pld_horario p ON p.data_hora = m.data_hora AND p.cod_submercado = u.cod_submercado
        WHERE u.cod_usina ~ '{COD_RE}'
        GROUP BY u.cod_usina, f.cod_fonte, u.cod_submercado, u.potencia_mw""")
    df["horas_ano"] = 8760
    return df

def agregar(df, por):
    g = df.groupby(por, dropna=False).agg(geracao_mwh=("geracao_mwh", "sum"), receita_rs=("receita_rs", "sum"),
                                          cap_mwh=("potencia_mw", lambda s: (s * 8760).sum())).reset_index()
    g["preco_capturado_rs_mwh"] = g["receita_rs"] / g["geracao_mwh"]
    g["fator_capacidade_pct"] = 100 * g["geracao_mwh"] / g["cap_mwh"]
    return g.drop(columns="cap_mwh")

ant, dep = indicadores(TABELAS_ANTES), indicadores(TABELAS_DEPOIS)
ant["total"] = "TOTAL"; dep["total"] = "TOTAL"

linhas = []
for nivel, col in (("total", "total"), ("fonte", "cod_fonte"), ("usina", "cod_usina")):
    a, d = agregar(ant, col).set_index(col), agregar(dep, col).set_index(col)
    for chave in a.index.union(d.index):
        for ind in ("geracao_mwh", "receita_rs", "preco_capturado_rs_mwh", "fator_capacidade_pct"):
            va = a.at[chave, ind] if chave in a.index else np.nan
            vd = d.at[chave, ind] if chave in d.index else np.nan
            delta = vd - va
            linhas.append((nivel, str(chave), ind, py(va), py(vd), py(delta),
                           py(100 * delta / va) if pd.notna(va) and va else None))

with conn.cursor() as cur:
    cur.execute("TRUNCATE qualidade.impacto_negocio")
    execute_values(cur, """INSERT INTO qualidade.impacto_negocio
                           (nivel, chave, indicador, antes, depois, delta, delta_pct) VALUES %s""", linhas)
conn.commit()

imp = consulta("SELECT * FROM qualidade.impacto_negocio")
def tabela(nivel):
    t = imp[imp["nivel"] == nivel].pivot(index="chave", columns="indicador", values=["antes", "depois", "delta_pct"])
    return t.round(2)

print("TOTAL (todas as usinas):")
display(tabela("total"))
print("\nPor fonte:")
display(tabela("fonte"))
print("\nPor usina (variação % de geração e de receita):")
display(imp[(imp["nivel"] == "usina") & imp["indicador"].isin(["geracao_mwh", "receita_rs"])]
        .pivot(index="chave", columns="indicador", values="delta_pct").round(2))

TOTAL (todas as usinas):


antes                                     \
indicador fator_capacidade_pct geracao_mwh preco_capturado_rs_mwh   
chave                                                               
TOTAL                    53.49  3865626.43                 207.96   

                                      depois              \
indicador    receita_rs fator_capacidade_pct geracao_mwh   
chave                                                      
TOTAL      8.039119e+08                43.16  3118868.66   

                                                          delta_pct  \
indicador preco_capturado_rs_mwh    receita_rs fator_capacidade_pct   
chave                                                                 
TOTAL                     204.45  6.376470e+08               -19.32   

                                                         
indicador geracao_mwh preco_capturado_rs_mwh receita_rs  
chave                                                    
TOTAL          -19.32                  -1.69     -20.68


Por fonte:


antes                                     \
indicador fator_capacidade_pct geracao_mwh preco_capturado_rs_mwh   
chave                                                               
BIO                     264.78   927792.19                 226.41   
EOL                      42.28   629673.66                 206.40   
PCH                      59.75   130846.39                 220.73   
UFV                      21.42   281523.13                 151.78   
UHE                      59.61  1514383.16                 201.28   
UTE                      29.03   381407.90                 229.29   

                                      depois              \
indicador    receita_rs fator_capacidade_pct geracao_mwh   
chave                                                      
BIO        2.100633e+08                52.44   183739.74   
EOL        1.299643e+08                42.25   629178.06   
PCH        2.888161e+07                59.72   130789.90   
UFV        4.273054e+07                21.39   281055.97   
UHE        3.048199e+08                59.57  1513225.09   
UTE        8.745229e+07                28.99   380879.90   

                                                          delta_pct  \
indicador preco_capturado_rs_mwh    receita_rs fator_capacidade_pct   
chave                                                                 
BIO                       240.43  4.417670e+07               -80.20   
EOL                       206.46  1.299017e+08                -0.08   
PCH                       220.80  2.887830e+07                -0.04   
UFV                       151.89  4.268954e+07                -0.17   
UHE                       201.37  3.047201e+08                -0.08   
UTE                       229.16  8.728056e+07                -0.14   

                                                         
indicador geracao_mwh preco_capturado_rs_mwh receita_rs  
chave                                                    
BIO            -80.20                   6.19     -78.97  
EOL             -0.08                   0.03      -0.05  
PCH             -0.04                   0.03      -0.01  
UFV             -0.17                   0.07      -0.10  
UHE             -0.08                   0.04      -0.03  
UTE             -0.14                  -0.06      -0.20


Por usina (variação % de geração e de receita):


indicador,geracao_mwh,receita_rs
chave,,
CRE-BIO-005,-80.20,-78.97
CRE-EOL-003,-0.37,-0.20
CRE-EOL-007,0.00,0.00
CRE-PCH-002,-0.04,-0.01
CRE-UFV-004,-0.31,-0.10
CRE-UFV-006,-0.07,-0.09
CRE-UHE-001,-0.12,-0.05
CRE-UHE-008,0.00,0.00
CRE-UTE-009,-0.14,-0.20


## 9. Índice de qualidade (para a pergunta 7 do painel)

In [15]:
VIEWS = """
CREATE OR REPLACE VIEW qualidade.vw_indice_qualidade AS
WITH ult AS (SELECT fase, MAX(id_execucao) AS id_execucao FROM qualidade.metricas_qualidade GROUP BY fase)
SELECT m.fase, m.dimensao, m.tabela,
       COUNT(*) AS regras,
       ROUND(AVG(100 - m.percentual_falha), 2) AS indice_qualidade_pct
FROM qualidade.metricas_qualidade m
JOIN ult u ON u.fase = m.fase AND u.id_execucao = m.id_execucao
WHERE m.erro IS NULL AND m.registros_avaliados > 0
GROUP BY m.fase, m.dimensao, m.tabela;

CREATE OR REPLACE VIEW qualidade.vw_indice_dimensao AS
WITH ult AS (SELECT fase, MAX(id_execucao) AS id_execucao FROM qualidade.metricas_qualidade GROUP BY fase)
SELECT m.fase, m.dimensao,
       COUNT(*) AS regras,
       ROUND(AVG(100 - m.percentual_falha), 2) AS indice_qualidade_pct
FROM qualidade.metricas_qualidade m
JOIN ult u ON u.fase = m.fase AND u.id_execucao = m.id_execucao
WHERE m.erro IS NULL AND m.registros_avaliados > 0
GROUP BY m.fase, m.dimensao;
"""
with conn.cursor() as cur:
    cur.execute(VIEWS)
conn.commit()

print("Índice por dimensão (100 = sem falhas):")
display(consulta("SELECT * FROM qualidade.vw_indice_dimensao ORDER BY dimensao, fase")
        .pivot(index="dimensao", columns="fase", values="indice_qualidade_pct"))

print("\nÍndice por tabela:")
display(consulta("""SELECT tabela, fase, ROUND(AVG(indice_qualidade_pct), 2) AS indice
                    FROM qualidade.vw_indice_qualidade GROUP BY tabela, fase ORDER BY tabela, fase""")
        .pivot(index="tabela", columns="fase", values="indice"))

Índice por dimensão (média simples do % de acerto das regras; 100 = sem falhas):


fase,ANTES,DEPOIS
dimensao,,
Acurácia,99.89,99.94
Atualidade,96.16,96.17
Completude,98.31,99.96
Consistência,96.97,100.00
Integridade referencial,99.99,100.00
Unicidade,96.65,100.00
Validade,96.36,100.00



Índice por tabela:


fase,ANTES,DEPOIS
tabela,,
clima_horario,100.00,100.00
medicao x clima,99.95,100.00
medicao x parada,99.99,100.00
medicao x usina,88.88,100.00
medicao_geracao,99.15,99.16
parada_usina,95.46,100.00
pld_horario,99.96,99.96
raw_pld_horario,100.00,100.00
usina,90.00,100.00
